<a href="https://colab.research.google.com/github/siva567-pixel/IntermediateAssessment-4/blob/main/Intermediate_assessment_3_jesna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicting the Sex of Laysan Albatross



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

# Find the competition CSV files
data_folder = Path("/content/drive/MyDrive/Colab Notebooks/DSAB11/intermediate_assessment_3")
train_files = list(data_folder.rglob("train.csv"))
test_files = list(data_folder.rglob("test.csv"))

print("Train files:", train_files)
print("Test files:", test_files)

if not train_files or not test_files:
    raise FileNotFoundError("Add the competition dataset using Kaggle's Add Input button.")

train = pd.read_csv(train_files[0])
test = pd.read_csv(test_files[0])

print("Training data:", train.shape)
print("Test data:", test.shape)
train.head()

Train files: [PosixPath('/content/drive/MyDrive/Colab Notebooks/DSAB11/intermediate_assessment_3/train.csv')]
Test files: [PosixPath('/content/drive/MyDrive/Colab Notebooks/DSAB11/intermediate_assessment_3/test.csv')]
Training data: (94, 14)
Test data: (41, 13)


,id,sexo,longitudCraneo,longitudPico,longitudNarina,anchoCraneo,altoPico,anchoPico,tarso,longAlaCerrada,longAlaAbierta,mediaEnvergadura,envergadura,peso
0,0J4_2016,H,174.18,106.85,80.19,52.43,30.20,23.26,88.74,51.0,95.7,104.2,208.4,2.38
1,A68_2016,H,171.12,104.59,79.45,51.14,30.61,24.04,90.22,50.5,94.4,102.2,204.4,2.14
2,E55_2015,H,174.69,106.83,82.14,49.09,30.60,26.47,91.42,52.7,96.6,105.2,210.4,2.50
3,7C7_2016,M,182.25,113.17,84.13,52.26,32.08,26.07,91.74,52.6,98.9,107.6,215.2,2.83
4,2C0_2018,H,173.70,105.19,80.37,49.20,34.00,27.12,87.77,50.8,94.5,103.7,207.4,2.73


## Check the data

In [ ]:
print(train["sexo"].value_counts())
print("\nMissing values:")
print(train.isnull().sum())

sexo
H    57
M    37
Name: count, dtype: int64

Missing values:
id                  0
sexo                0
longitudCraneo      0
longitudPico        0
longitudNarina      0
anchoCraneo         0
altoPico            0
anchoPico           0
tarso               0
longAlaCerrada      0
longAlaAbierta      0
mediaEnvergadura    0
envergadura         0
peso                1
dtype: int64


## Prepare and validate the model

In [ ]:
features = [col for col in train.columns if col not in ["id", "sexo"]]
X = train[features]
y = train["sexo"]
X_test = test[features]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# SVM works better when the numeric features are scaled
model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    SVC(C=10, kernel="rbf", probability=True, random_state=42)
)

model.fit(X_train, y_train)

m_index = list(model.classes_).index("M")
valid_prob = model.predict_proba(X_valid)[:, m_index]

# This threshold is an experiment; compare the leaderboard result
valid_pred = ["M" if p >= 0.40 else "H" for p in valid_prob]

print("Validation accuracy:", round(accuracy_score(y_valid, valid_pred), 3))
print("Validation macro F1:", round(f1_score(y_valid, valid_pred, average="macro"), 3))
print("Validation F1 for M:", round(f1_score(y_valid, valid_pred, pos_label="M"), 3))

Validation accuracy: 0.789
Validation macro F1: 0.756
Validation F1 for M: 0.667


## Train on all labeled data

In [ ]:
model.fit(X, y)

m_index = list(model.classes_).index("M")
test_prob = model.predict_proba(X_test)[:, m_index]
test_pred = ["M" if p >= 0.40 else "H" for p in test_prob]

print("Number of predictions:", len(test_pred))
print(pd.Series(test_pred).value_counts())

Number of predictions: 41
M    22
H    19
Name: count, dtype: int64


## Create the Kaggle submission

In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    "sexo": test_pred
})

submission.to_csv("/content/drive/MyDrive/Colab Notebooks/DSAB11/intermediate_assessment_3/submission.csv", index=False)
print("Submission shape:", submission.shape)
submission.head()

Submission shape: (41, 2)


,id,sexo
0,A70_2018,M
1,5C9_2018,H
2,5E1_2018,M
3,C58_2018,M
4,5E2_2018,M
